# **Q-04: TẠO CACHE LOG-MEL SPECTROGRAM CHO TOÀN BỘ DỮ LIỆU**

## **Import thư viện**

In [3]:
import torch
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm  # thanh tiến trình
import sys

## **Kết nối drive & Import wav_to_logmel**

In [4]:
from google.colab import drive
drive.mount('/content/drive')

sys.path.append('/content/drive/MyDrive/esc50')
sys.path.append('/content/drive/MyDrive/esc50/data/ESC-50-master')

from src.data_lib import wav_to_logmel

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## **Cấu hình**

In [5]:
DATA_DIR = Path("/content/drive/MyDrive/esc50/data/ESC-50-master")
AUDIO_DIR = DATA_DIR / "audio"
META_PATH = DATA_DIR / "meta" / "esc50.csv"
CACHE_DIR = DATA_DIR / "cache"
CACHE_DIR.mkdir(exist_ok=True)

## **Đọc metadata**

In [6]:
df = pd.read_csv(META_PATH)
print(f"Có {len(df)} mẫu từ esc50.csv")

classes = df.groupby('target')['category'].first().tolist()
print(f"Có {len(classes)} lớp âm thanh")

Có 2000 mẫu từ esc50.csv
Có 50 lớp âm thanh


## **Hàm cache**

In [7]:
def build_cache(n_mels: int, device: str = "cpu"):
    """
    Tạo file cache Log-Mel cho toàn bộ 2000 mẫu.
    n_mels: 64 hoặc 128
    """
    print(f"\n{'='*60}")
    print(f"Bắt đầu tạo cache: n_mels = {n_mels}")
    print(f"{'='*60}")

    X = torch.empty((len(df), 1, n_mels, 216), dtype=torch.float32)
    y = torch.tensor(df['target'].values, dtype=torch.int64)
    fold = torch.tensor(df['fold'].values, dtype=torch.int64)
    filenames = df['filename'].tolist()

    for idx, row in tqdm(df.iterrows(), total=len(df), desc=f"Mel {n_mels}"):
        wav_path = AUDIO_DIR / row['filename']
        X[idx] = wav_to_logmel(str(wav_path), n_mels=n_mels, device=device)

    cache_data = {
        'X': X,
        'y': y,
        'fold': fold,
        'filenames': filenames,
        'classes': classes
    }

    cache_path = CACHE_DIR / f"logmel{n_mels}.pt"
    torch.save(cache_data, cache_path)

    size_mb = cache_path.stat().st_size / (1024 * 1024)
    print(f"Đã lưu cache: {cache_path}")
    print(f"Shape X: {tuple(X.shape)}")
    print(f"Kích thước file: {size_mb:.1f} MB")

    return cache_path

## **Cache 64 Mel bins**

In [8]:
cache64 = build_cache(n_mels=64, device="cpu")


Bắt đầu tạo cache: n_mels = 64


Mel 64:   0%|          | 0/2000 [00:00<?, ?it/s]

Đã lưu cache: /content/drive/MyDrive/esc50/data/ESC-50-master/cache/logmel64.pt
Shape X: (2000, 1, 64, 216)
Kích thước file: 105.6 MB


## **Cache 128 Mel bins**

In [9]:
cache128 = build_cache(n_mels=128, device="cpu")


Bắt đầu tạo cache: n_mels = 128


Mel 128:   0%|          | 0/2000 [00:00<?, ?it/s]

Đã lưu cache: /content/drive/MyDrive/esc50/data/ESC-50-master/cache/logmel128.pt
Shape X: (2000, 1, 128, 216)
Kích thước file: 211.0 MB


## **Kiểm tra**

In [10]:
print(f"\n{'='*60}")
print("CACHE HOÀN THÀNH!")
print(f"{cache64}")
print(f"{cache128}")
print(f"{'='*60}")


CACHE HOÀN THÀNH!
/content/drive/MyDrive/esc50/data/ESC-50-master/cache/logmel64.pt
/content/drive/MyDrive/esc50/data/ESC-50-master/cache/logmel128.pt
